# Draft-03: XGBoost Fraud Pipeline (Kaggle Runtime)

Main goal: maximize validation F1 score for highly imbalanced fraud detection.

In [ ]:
import importlib.util
import subprocess
import sys


def run_cmd(cmd):
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


def ensure_package(module_name, package_name=None):
    pkg = package_name or module_name
    if importlib.util.find_spec(module_name) is not None:
        print(f"{pkg} is already installed")
        return
    run_cmd([sys.executable, "-m", "pip", "install", pkg])


ensure_package("xgboost")
ensure_package("dotenv", "python-dotenv")
ensure_package("kaggle")

from xgboost import XGBClassifier

print("Bootstrap dependencies are ready")

Install instruction (run once if your Kaggle image is missing packages):

```bash
pip install xgboost python-dotenv kaggle
```

For Kaggle API download/submit, add your `KAGGLE_USERNAME` and `KAGGLE_KEY` in Kaggle notebook secrets (or `.env`).

In [ ]:
import os
from datetime import datetime
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from kaggle.api.kaggle_api_extended import KaggleApi
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

COMPETITION = "fraud-detection-cpe-232-data-models"
RUN_DOWNLOAD = True
RUN_SUBMISSION = True

ID_COL = "id"
LABEL_COL = "is_fraud"
TIME_COL = "time_ind"
RANDOM_STATE = 42
VALID_SIZE = 0.20
EPS = 1.0


def resolve_data_paths_fallback():
    kaggle_candidates = [
        Path("/kaggle/input/fraud-detection-cpe-232-data-models"),
        Path("/kaggle/input/fraudulent-transaction-detect"),
        Path("/kaggle/input"),
    ]
    local_candidates = [
        Path.cwd(),
        Path.cwd() / "kaggle/01-fraudulent-transaction-detect",
        Path(
            "/Users/beam/Workspace/Course/my-cpe-lab/Y2/CPE232/kaggle/01-fraudulent-transaction-detect"
        ),
    ]

    candidate_pairs = []
    for base in kaggle_candidates + local_candidates:
        candidate_pairs.extend(
            [
                (base / "train.csv", base / "test.csv", base / "sample_submission.csv"),
                (
                    base / "data" / "train.csv",
                    base / "data" / "test.csv",
                    base / "data" / "sample_submission.csv",
                ),
                (
                    base / "fraud-detection-cpe-232-data-models" / "train.csv",
                    base / "fraud-detection-cpe-232-data-models" / "test.csv",
                    base
                    / "fraud-detection-cpe-232-data-models"
                    / "sample_submission.csv",
                ),
            ]
        )
    for train_path, test_path, sample_path in candidate_pairs:
        if train_path.exists() and test_path.exists() and sample_path.exists():
            return train_path, test_path, sample_path
    raise FileNotFoundError(
        "Could not find train/test/sample CSV files in Kaggle input or local folders"
    )


def prepare_competition_data(api_client, competition, data_dir, force_download=False):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / f"{competition}.zip"
    extract_dir = data_dir / competition

    if force_download or not zip_path.exists():
        files_resp = api_client.competition_list_files(competition)
        try:
            api_client.competition_download_files(
                competition,
                path=str(data_dir),
                force=force_download,
                quiet=False,
            )
            print("Competition zip download complete")
        except Exception as download_error:
            print(
                "Bulk download failed, fallback to per-file download:", download_error
            )
            for f in files_resp.files:
                file_path = data_dir / f.name
                if file_path.exists() and not force_download:
                    continue
                api_client.competition_download_file(
                    competition=competition,
                    file_name=f.name,
                    path=str(data_dir),
                    force=force_download,
                    quiet=False,
                )

    if zip_path.exists():
        if not extract_dir.exists() or not any(extract_dir.iterdir()):
            extract_dir.mkdir(parents=True, exist_ok=True)
            with ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)

    train_path = extract_dir / "train.csv"
    test_path = extract_dir / "test.csv"
    sample_path = extract_dir / "sample_submission.csv"

    if not (train_path.exists() and test_path.exists() and sample_path.exists()):
        fallback_train = data_dir / "train.csv"
        fallback_test = data_dir / "test.csv"
        fallback_sample = data_dir / "sample_submission.csv"
        if (
            fallback_train.exists()
            and fallback_test.exists()
            and fallback_sample.exists()
        ):
            train_path, test_path, sample_path = (
                fallback_train,
                fallback_test,
                fallback_sample,
            )
        else:
            raise FileNotFoundError(
                "Could not resolve competition train/test/sample files"
            )

    return train_path, test_path, sample_path, zip_path, extract_dir


load_dotenv(".env", override=True)
os.environ.pop("KAGGLE_API_TOKEN", None)

api = None
try:
    api = KaggleApi()
    api.authenticate()
    print("Authenticated user:", api.get_config_value("username"))
except Exception as auth_error:
    print("Kaggle API auth not ready:", auth_error)
    print("Falling back to existing /kaggle/input or local data if available")

if RUN_DOWNLOAD:
    if api is None:
        raise RuntimeError("RUN_DOWNLOAD=True but Kaggle API auth is not available")
    BASE_DIR = Path.cwd()
    DATA_DIR = BASE_DIR / "data"
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH, ZIP_PATH, EXTRACT_DIR = (
        prepare_competition_data(
            api_client=api,
            competition=COMPETITION,
            data_dir=DATA_DIR,
            force_download=False,
        )
    )
else:
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH = resolve_data_paths_fallback()
    BASE_DIR = TRAIN_PATH.parent

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else BASE_DIR
OUTPUT_PATH = OUTPUT_DIR / "submission_draft03_xgboost_kaggle.csv"

print("Competition:", COMPETITION)
print("RUN_DOWNLOAD:", RUN_DOWNLOAD)
print("RUN_SUBMISSION:", RUN_SUBMISSION)
print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)
print("Sample path:", SAMPLE_PATH)
print("Output path:", OUTPUT_PATH)

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

expected_train_cols = {
    "id",
    "time_ind",
    "transac_type",
    "amount",
    "src_acc",
    "src_bal",
    "src_new_bal",
    "dst_acc",
    "dst_bal",
    "dst_new_bal",
    "is_fraud",
    "is_flagged_fraud",
}
expected_test_cols = expected_train_cols - {LABEL_COL}

assert set(train_df.columns) == expected_train_cols, "Train schema mismatch"
assert set(test_df.columns) == expected_test_cols, "Test schema mismatch"
assert sample_submission.columns.tolist() == [ID_COL, LABEL_COL], (
    "Sample submission schema mismatch"
)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
class_counts = train_df[LABEL_COL].value_counts().sort_index()
fraud_rate = float(train_df[LABEL_COL].mean())
print("\nClass counts (0, 1):")
print(class_counts.to_string())
print(f"Fraud rate: {fraud_rate:.8f} ({fraud_rate * 100:.4f}%)")

display(train_df.head(3))

## Feature Engineering (Leakage-safe)

In [ ]:
NUMERIC_SOURCE_COLS = ["amount", "src_bal", "src_new_bal", "dst_bal", "dst_new_bal"]


def get_fill_values(fit_df: pd.DataFrame) -> dict[str, float]:
    medians = fit_df[NUMERIC_SOURCE_COLS].median(numeric_only=True).to_dict()
    return {k: float(v) for k, v in medians.items()}


def build_account_maps(fit_df: pd.DataFrame) -> dict[str, pd.Series]:
    src_group = fit_df.groupby("src_acc", observed=True).agg(
        src_txn_count=(ID_COL, "size"),
        src_unique_dst=("dst_acc", "nunique"),
    )
    dst_group = fit_df.groupby("dst_acc", observed=True).agg(
        dst_txn_count=(ID_COL, "size"),
        dst_unique_src=("src_acc", "nunique"),
    )
    return {
        "src_txn_count": src_group["src_txn_count"],
        "src_unique_dst": src_group["src_unique_dst"],
        "dst_txn_count": dst_group["dst_txn_count"],
        "dst_unique_src": dst_group["dst_unique_src"],
    }


def build_base_features(
    df: pd.DataFrame, fill_values: dict[str, float]
) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    amount = df["amount"].fillna(fill_values["amount"]).astype("float64")
    src_bal = df["src_bal"].fillna(fill_values["src_bal"]).astype("float64")
    src_new_bal = df["src_new_bal"].fillna(fill_values["src_new_bal"]).astype("float64")
    dst_bal = df["dst_bal"].fillna(fill_values["dst_bal"]).astype("float64")
    dst_new_bal = df["dst_new_bal"].fillna(fill_values["dst_new_bal"]).astype("float64")
    time_ind = df[TIME_COL].astype("int64")

    out["transac_type"] = df["transac_type"].fillna("MISSING").astype(str)
    out["amount"] = amount
    out["src_bal"] = src_bal
    out["src_new_bal"] = src_new_bal
    out["dst_bal"] = dst_bal
    out["dst_new_bal"] = dst_new_bal
    out["is_flagged_fraud"] = df["is_flagged_fraud"].astype("float64")

    out["src_balance_diff"] = src_new_bal - src_bal
    out["dst_balance_diff"] = dst_new_bal - dst_bal

    out["expected_src_new"] = src_bal - amount
    out["src_error"] = out["expected_src_new"] - src_new_bal
    out["expected_dst_new"] = dst_bal + amount
    out["dst_error"] = out["expected_dst_new"] - dst_new_bal

    out["src_zero_before"] = (src_bal == 0).astype("float64")
    out["src_zero_after"] = (src_new_bal == 0).astype("float64")
    out["dst_zero_before"] = (dst_bal == 0).astype("float64")
    out["dst_zero_after"] = (dst_new_bal == 0).astype("float64")

    out["amount_to_src_balance"] = amount / (np.abs(src_bal) + EPS)
    out["amount_to_dst_balance"] = amount / (np.abs(dst_bal) + EPS)
    out["amount_to_src_new_balance"] = amount / (np.abs(src_new_bal) + EPS)

    out["emptied_account"] = ((src_bal > 0) & (src_new_bal == 0)).astype("float64")
    out["large_transfer"] = (amount > 200_000).astype("float64")

    out["hour"] = (time_ind % 24).astype("float64")
    out["day"] = (time_ind // 24).astype("float64")
    out["is_night"] = out["hour"].isin([0, 1, 2, 3, 4, 5]).astype("float64")
    out["log_amount"] = np.log1p(amount)

    src_prefix_merchant = df["src_acc"].astype(str).str.lower().str.startswith("mer")
    dst_prefix_merchant = df["dst_acc"].astype(str).str.lower().str.startswith("mer")
    prefix_merchant = src_prefix_merchant | dst_prefix_merchant
    missing_merchant_proxy = df[["dst_bal", "dst_new_bal"]].isna().any(axis=1)
    out["merchant_prefix_flag"] = prefix_merchant.astype("float64")
    out["merchant_missing_proxy"] = missing_merchant_proxy.astype("float64")
    out["merchant_indicator"] = np.where(
        prefix_merchant, 1.0, missing_merchant_proxy.astype(float)
    )

    return out


def add_account_features(
    base_features: pd.DataFrame, df: pd.DataFrame, account_maps: dict[str, pd.Series]
) -> pd.DataFrame:
    out = base_features.copy()
    out["src_txn_count"] = (
        df["src_acc"].map(account_maps["src_txn_count"]).fillna(0.0).astype("float64")
    )
    out["src_unique_dst"] = (
        df["src_acc"].map(account_maps["src_unique_dst"]).fillna(0.0).astype("float64")
    )
    out["dst_txn_count"] = (
        df["dst_acc"].map(account_maps["dst_txn_count"]).fillna(0.0).astype("float64")
    )
    out["dst_unique_src"] = (
        df["dst_acc"].map(account_maps["dst_unique_src"]).fillna(0.0).astype("float64")
    )
    return out


def make_model_matrix(
    fit_features: pd.DataFrame,
    transform_features: pd.DataFrame,
    encoder: ColumnTransformer | None = None,
):
    categorical_cols = ["transac_type"]
    numeric_cols = [c for c in fit_features.columns if c not in categorical_cols]
    if encoder is None:
        encoder = ColumnTransformer(
            transformers=[
                (
                    "transac_type_ohe",
                    OneHotEncoder(handle_unknown="ignore", sparse_output=True),
                    categorical_cols,
                ),
                ("numeric", "passthrough", numeric_cols),
            ],
            remainder="drop",
            sparse_threshold=1.0,
        )
        fit_matrix = encoder.fit_transform(fit_features)
    else:
        fit_matrix = encoder.transform(fit_features)
    transform_matrix = encoder.transform(transform_features)
    return fit_matrix, transform_matrix, encoder


def find_best_f1_threshold(y_true: pd.Series, y_prob: np.ndarray):
    thresholds = np.linspace(0.001, 0.999, 999)
    best_t = 0.5
    best_f1 = -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_t = float(t)
    return best_t, float(best_f1)


def has_inf_values(df: pd.DataFrame) -> bool:
    numeric_df = df.select_dtypes(include=[np.number])
    for col in numeric_df.columns:
        if np.isinf(numeric_df[col].to_numpy()).any():
            return True
    return False

In [ ]:
train_fit_df, valid_df = train_test_split(
    train_df,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    stratify=train_df[LABEL_COL],
)
train_fit_df = train_fit_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

fill_values = get_fill_values(train_fit_df)
account_maps = build_account_maps(train_fit_df)

X_fit_base = build_base_features(train_fit_df, fill_values)
X_valid_base = build_base_features(valid_df, fill_values)
X_fit = add_account_features(X_fit_base, train_fit_df, account_maps)
X_valid = add_account_features(X_valid_base, valid_df, account_maps)
y_fit = train_fit_df[LABEL_COL].astype(int)
y_valid = valid_df[LABEL_COL].astype(int)

assert not has_inf_values(X_fit), "Found inf values in training features"
assert not has_inf_values(X_valid), "Found inf values in validation features"

X_fit_matrix, X_valid_matrix, encoder = make_model_matrix(X_fit, X_valid, encoder=None)

valid_probe = X_valid.head(8).copy()
valid_probe.loc[:, "transac_type"] = "UNSEEN_TYPE"
_ = encoder.transform(valid_probe)
print("Unseen transac_type handling check: passed")

pos_count = int(y_fit.sum())
neg_count = int(len(y_fit) - pos_count)
scale_pos_weight = float(neg_count / max(pos_count, 1))
print("train_fit shape:", X_fit.shape)
print("valid shape:", X_valid.shape)
print("train_fit fraud rate:", float(y_fit.mean()))
print("valid fraud rate:", float(y_valid.mean()))
print("scale_pos_weight:", scale_pos_weight)

In [ ]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=120,
)

xgb_model.fit(
    X_fit_matrix,
    y_fit,
    eval_set=[(X_valid_matrix, y_valid)],
    verbose=False,
)

valid_prob = xgb_model.predict_proba(X_valid_matrix)[:, 1]
f1_at_05 = f1_score(y_valid, (valid_prob >= 0.5).astype(int), zero_division=0)
best_threshold, best_valid_f1 = find_best_f1_threshold(y_valid, valid_prob)

best_iter = getattr(xgb_model, "best_iteration", None)
if best_iter is None:
    best_iter = xgb_model.get_params()["n_estimators"] - 1
best_n_estimators = int(best_iter + 1)

print(f"F1 @ 0.5: {f1_at_05:.6f}")
print(f"Best threshold: {best_threshold:.3f}")
print(f"Best validation F1: {best_valid_f1:.6f}")
print(f"Best iteration used for final fit: {best_n_estimators}")

In [ ]:
full_fill_values = get_fill_values(train_df)
full_account_maps = build_account_maps(train_df)

X_full_base = build_base_features(train_df, full_fill_values)
X_test_base = build_base_features(test_df, full_fill_values)
X_full = add_account_features(X_full_base, train_df, full_account_maps)
X_test = add_account_features(X_test_base, test_df, full_account_maps)

assert not has_inf_values(X_full), "Found inf values in full training features"
assert not has_inf_values(X_test), "Found inf values in test features"

X_full_matrix, X_test_matrix, final_encoder = make_model_matrix(
    X_full, X_test, encoder=None
)
y_full = train_df[LABEL_COL].astype(int)
pos_full = int(y_full.sum())
neg_full = int(len(y_full) - pos_full)
scale_pos_full = float(neg_full / max(pos_full, 1))

final_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_estimators=best_n_estimators,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    scale_pos_weight=scale_pos_full,
)

final_model.fit(X_full_matrix, y_full, verbose=False)
test_prob = final_model.predict_proba(X_test_matrix)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

submission = pd.DataFrame({ID_COL: test_df[ID_COL], LABEL_COL: test_pred})
submission.to_csv(OUTPUT_PATH, index=False)

summary = pd.DataFrame(
    [
        {
            "train_rows": int(len(train_df)),
            "valid_rows": int(len(valid_df)),
            "fraud_rate_train": float(train_df[LABEL_COL].mean()),
            "best_threshold": float(best_threshold),
            "best_valid_f1": float(best_valid_f1),
            "f1_at_0.5": float(f1_at_05),
            "predicted_fraud_rows_test": int(submission[LABEL_COL].sum()),
            "predicted_fraud_rate_test": float(submission[LABEL_COL].mean()),
        }
    ]
)
display(summary)
print("Saved submission to", OUTPUT_PATH)

In [ ]:
if RUN_SUBMISSION:
    if api is None:
        raise RuntimeError("RUN_SUBMISSION=True but Kaggle API auth is not available")

    submit_message = (
        f"draft-03 xgboost best_f1={best_valid_f1:.6f} "
        f"threshold={best_threshold:.3f} "
        f"time={datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')}Z"
    )
    response = api.competition_submit(
        file_name=str(OUTPUT_PATH),
        message=submit_message,
        competition=COMPETITION,
        quiet=False,
    )
    print("Submission response:", response)
else:
    print("RUN_SUBMISSION is False. File is ready at", OUTPUT_PATH)

## Notes

- This notebook supports Kaggle API authenticate/download/submit workflow.
- `RUN_DOWNLOAD=True` downloads/extracts competition data into notebook working folder.
- `RUN_SUBMISSION=False` by default to avoid accidental submits; set it to `True` to auto-submit.
- Threshold is optimized on validation probabilities to maximize F1.
- `transac_type` uses one-hot with `handle_unknown="ignore"`.
- Account frequency features are built from train-fit data only for validation stage.